# 15 CBS：冲突树最优 MAPF


本课按百科条目写。CBS 是离散 MAPF 求 SOC 最优的主流精确算法。调度台在最后。

---

## 1. 定义

**Conflict-Based Search**（Sharon, Stern, Felner, Sturtevant, AIJ 2015）：两层搜索。

- **高层**：冲突树 CT。每个节点 = 约束集合 + 一组单车路径。
- **底层**：对某个智能体，在该节点的约束下做时间 A\*（本仓库；也可以 SIPP）。

根节点：各车 **独立** 最短路（普通 A\*，互不理睬）。若两路径在 (点, t) 或换边冲突，分裂：

- 左孩子：禁止 A 在该 (点, t)（或禁止走那条边拍）
- 右孩子：禁止 B 在该 (点, t)

高层按 SOC（各车到达时间和）做最佳优先。弹出的 **第一个无冲突叶** 即经典离散 MAPF 的 SOC 最优解。

- 输入：图、各车起终点、高层 cap。
- 输出：无冲突路径字典，SOC，高层/底层扩展数。

---

## 2. 和 HCA* / 联合 A*

| | 联合 A\* | HCA\* | CBS |
|--|--|--|--|
| 状态 | 联合位置 | 单车 + 全序 | 单车 + 约束集 |
| 最优 | 是 | 否 | 是（SOC） |
| 分支 | \(5^k\) | 1 条全序 | 每个冲突 2 叉 |
| 何时快 | k≤2 | 几乎总是 | 冲突少时远快于联合 A\* |

CBS 只在 **真发生的冲突** 上分支，底层仍是单车搜索。冲突少（空旷、错开）时树很小；全员挤窄廊时树仍指数。MAPF 是 NP-hard，最坏躲不掉。

---

## 3. 不变量（为何最优）

- CT 节点的约束只增不减。
- 底层路径满足该节点全部约束（约束写成占用表里的 block）。
- **可采纳：** 节点的 SOC = 各车在 **当前约束下** 的最优代价之和，是该支任何可行解的下界（再加约束只会更差或持平）。
- 高层堆先弹 SOC 最小者。第一个可行叶：不存在 SOC 更小的可行解，否则它的下界更小，应先被弹出。

stay-at-target：先到的车停在终点，后面的拍仍占该格。`vertex_conflicts` 用 `pad` 把短路径补成终点停留再比。

---

## 4. 伪代码

```
root.paths ← 各车 A*
root.constraints ← ∅
push (SOC(root), root)
while heap:
    N ← pop 最小 SOC
    if N.paths 无冲突: return N.paths
    (a,b,v,t) ← 第一冲突
    for who in {a,b}:
        C ← N.constraints ∪ {who 不能在 (v,t)}
        π ← TimeA*(who, C)
        若成功: 更新该车路径，push 新节点
```

边冲突类似，约束写成禁止走某有向边拍。`cap` 限制高层弹出次数，无解时避免死循环。

---

## 5. 手算

1 宽走廊两车对换。独立最短路在中点同一拍相遇。

| CT 节点 | 约束 | 路径要点 | SOC |
|---------|------|----------|-----|
| 根 | ∅ | 对头，中点冲突 | 2×长度 |
| 左 | a 不能占中点@t* | a 等一拍 | +1 |
| 右 | b 不能占中点@t* | b 等一拍 | +1 |

任一孩子无冲突即可返回。最优 SOC = 独立和 + 1（一人等）。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 6. 实现：冲突检测 + 两车 CBS


In [ ]:
from lib.agents import pad, soc, vertex_conflicts, edge_conflicts, is_conflict_free


def first_conflict(paths):
    # 先顶点、后换边。返回 ('vertex', a, b, loc, t) 或 ('edge', a, b, (u, v), t)
    vc = vertex_conflicts(paths)
    if vc:
        a, b, loc, t = vc[0]
        return ("vertex", a, b, loc, t)
    ec = edge_conflicts(paths)
    if ec:
        a, b, u, v, t = ec[0]
        return ("edge", a, b, (u, v), t)
    return None


paths = {"a": [(0, 1), (1, 1), (2, 1)], "b": [(2, 1), (1, 1), (0, 1)]}
print("手写冲突", first_conflict(paths))
print("库顶点", vertex_conflicts(paths))
print("库换边", edge_conflicts(paths))
cross = {"a": [(0, 0), (1, 0)], "b": [(1, 0), (0, 0)]}
print("对向穿过（无同格、有换边）", first_conflict(cross), edge_conflicts(cross))


In [ ]:
from lib.search import astar
from lib.algos.temporal import time_astar
from lib.reservation import ReservationTable
from evals.scenarios import two_agent_swap


def replan(graph, start, goal, constraints, agent):
    # 约束 (kind, loc, t, who)；只执行属于自己的
    table = ReservationTable()
    for kind, loc, t, who in constraints:
        if who != agent:
            continue
        if kind == "vertex":
            table.occupy(loc, t, "block")
        elif kind == "edge":
            a, b = loc
            table.occupy_edge(a, b, t, "block")
    return time_astar(graph, start, goal, table=table, agent=agent)


def cbs_two(graph, starts, goals, cap=80, verbose=True):
    agents = list(starts)
    paths = {}
    for a in agents:
        r = astar(graph, starts[a], goals[a], record=False)
        if not r.found:
            return {"found": False, "paths": {}}
        paths[a] = r.path
    nodes = [(soc(paths), [], paths)]
    high = 0
    while nodes and high < cap:
        nodes.sort(key=lambda x: x[0])
        cost, cons, paths = nodes.pop(0)
        high += 1
        conf = first_conflict(paths)
        if verbose:
            print(f"高层#{high} SOC={cost} 约束数={len(cons)} 冲突{conf}")
        if conf is None:
            return {"found": True, "paths": paths, "soc": cost, "high": high}
        kind, a, b, loc, t = conf
        if kind == "vertex":
            branches = [("vertex", loc, t, a), ("vertex", loc, t, b)]
        else:
            u, v = loc
            branches = [("edge", (u, v), t, a), ("edge", (v, u), t, b)]
        for ck, cloc, ct, who in branches:
            new_cons = cons + [(ck, cloc, ct, who)]
            new_paths = dict(paths)
            r = replan(graph, starts[who], goals[who], new_cons, who)
            if not r.found:
                continue
            new_paths[who] = r.path
            nodes.append((soc(new_paths), new_cons, new_paths))
    return {"found": False, "paths": {}, "high": high}


g, starts, goals = two_agent_swap()
r = cbs_two(g, starts, goals)
print("found", r["found"], "SOC", r.get("soc"), "高层", r.get("high"))
print("无冲突", is_conflict_free(r["paths"]) if r["found"] else None)
print("a", r["paths"].get("a"))
print("b", r["paths"].get("b"))


## 7. 逐行

| 块 | 作用 |
|----|------|
| 根节点 `astar` 各车 | 无约束最短路，下界 |
| `first_conflict` | 先顶点、后换边；漏换边会宣布「无冲突」其实对向穿过 |
| `replan` | 约束 → 占用表 block → 时间 A\* |
| 两叉 | 顶点禁同一 (点,t)；换边禁 (u,v,t) 与 (v,u,t) |
| 按 SOC 弹出 | 高层可采纳 |

库 `cbs` 同样处理边冲突、`cap`、增强开关。只查顶点时，两车对换会在「各走各的下一格」漏检。


In [ ]:
from lib.algos.cbs import cbs

lib_r = cbs(g, starts, goals)
print("库 found", lib_r["found"], "SOC", lib_r["soc"], "高层", lib_r["high"], "底层扩展", lib_r["low"])


## 常见 bug

1. 底层不用时间维，无法等待消冲突。
2. 冲突只拆一次就停，没把新路径再送回高层。
3. stay-at-target 与 disappear 混用，终点占用不一致。
4. 约束没写进表，底层仍走原路，树无限长。
5. 无 cap：无解实例（两车换位却只有一格）会转到内存耗尽。
6. 只查顶点、不查换边：两车对向穿过会被当成无冲突。

## 条目小结

| 项目 | 内容 |
|------|------|
| 高层 | 冲突树 + SOC 最佳优先 |
| 底层 | 约束下的时间 A\* |
| 保证 | SOC 最优（经典离散 MAPF） |
| 下一课 | 少分支：基数冲突与 bypass |

## 练习

1. 根节点 SOC 为什么是下界？加约束后 SOC 会变小吗？
2. 用纸画出 two_agent_swap 根节点两条路径，标出第一冲突的 (点,t)。
3. 若只分支 a、不分支 b，还保证最优吗？


In [ ]:
from lib.studio.widget import PlannerStudio
studio = PlannerStudio()
studio.show_multi(g.to_studio(), lib_r, "CBS")
studio
